<a href="https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ML-10 Setup
# This regenerates the ranked refresh queue from the bundled starter CSV by re-running
# the repo's own reference pipeline (scripts/01-04) -- same feature prep, same baseline
# rule, same client-holdout model training as ML-07/ML-08/ML-09. Nothing here is hand-typed:
# every number the rest of this notebook cites comes out of this run.

import json
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO_URL = "https://github.com/krithi-ks/flyrankAI-ML.git"
REPO_DIR = Path("/content/flyrankAI-ML")  # Colab's cwd is /content, not the repo

candidate_roots = [Path("."), Path(".."), Path("../.."), REPO_DIR]
ROOT = next((p for p in candidate_roots if (p / "scripts" / "run_all.py").exists()), None)
if ROOT is None:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    ROOT = REPO_DIR
ROOT = ROOT.resolve()
print(f"Repo root: {ROOT}")

# Re-run the reference pipeline (skip step 5, the PDF export -- not needed here and
# avoids a reportlab dependency for a notebook-only run).
for step in ["01_prepare_features.py", "02_baseline_score.py", "03_train_model.py", "04_evaluate_and_export.py"]:
    subprocess.run([sys.executable, str(ROOT / "scripts" / step)], cwd=str(ROOT), check=True)

queue = pd.read_csv(ROOT / "outputs" / "refresh_queue.csv")
model_results = json.loads((ROOT / "outputs" / "model_results.json").read_text())
raw = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")

print(f"\nRanked queue rows: {len(queue):,}")
print(f"Best model: {model_results['best_model']['name']}  (selected by {model_results['best_model']['selection_metric']})")

Repo root: /content/flyrankAI-ML

Ranked queue rows: 30,000
Best model: random_forest  (selected by precision_at_50)


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue reuses the reason-code and action logic already built and tested in ML-07/ML-10's pipeline (`scripts/02_baseline_score.py` + `scripts/04_evaluate_and_export.py`) rather than inventing a new rubric here. Two layers combine:

- **Rule-layer reason codes** (transparent, hand-written): `stale_visible_page`, `declining_with_demand`, `thin_visible_page`, `page_one_decay_risk`, `low_ctr_visible_page`, `low_engagement_visible_page`.
- **Model-layer reason codes** (added when the random-forest probability clears a threshold): `model_decline_risk`, `visible_model_opportunity`, `ctr_review_candidate`, `engagement_review_candidate`.

Every row keeps its full trail of reasons, and `final_refresh_score` blends both layers (70% model probability, 30% normalized baseline rule score) so a reviewer sees the *why*, not just a number.

**Archetype → action mapping** (read directly off the pipeline code, not re-derived by eye):

| Signal pattern observed | Action assigned |
|---|---|
| `thin_visible_page` (visible, page is short) | `expand_and_refresh` |
| `low_ctr_visible_page` **and** (`model_decline_risk` or `declining_with_demand`) | `refresh_and_review_ctr` |
| `engagement_review_candidate` **and** (`model_decline_risk` or `declining_with_demand`) | `refresh_and_review_engagement` |
| any of `model_decline_risk` / `declining_with_demand` / `stale_visible_page` / `visible_model_opportunity`, no sharper match above | `refresh` |
| none of the above cleared | `monitor` |

**Decay/refresh insight — stated carefully.** I checked whether staler content actually shows more decline in this data (the intuitive "content rot" story). It does **not** show a clean pattern here: the `181+`-day freshness tier has the *lowest* observed decline share (47.1%), not the highest, and the `91-180` tier has the highest (61.1%) — see the table below. I'm reporting this as observed, not as a confirmed mechanism: with one 90-day snapshot and no before/after refresh outcome in this dataset, I can't tell whether this is a real non-monotonic relationship, a confound (e.g. older pages skew toward already-stable evergreen content), or noise from uneven bucket sizes. The honest takeaway is: **don't use `days_since_last_update` alone as a refresh trigger** — it's one of several signals the model and rules already weigh, not a standalone rule of thumb.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

action_counts = queue["suggested_action"].value_counts()
confidence_counts = queue["confidence"].value_counts().reindex(["high", "medium", "low"])
action_by_confidence = pd.crosstab(queue["suggested_action"], queue["confidence"])[["high", "medium", "low"]]

print(f"Suggested-action mix (n = {len(queue):,} scored rows):")
display(action_counts)

print("\nConfidence mix:")
display(confidence_counts)

print("\nAction x confidence (counts):")
display(action_by_confidence)

decline_rate_by_conf = queue.groupby("confidence")["is_declining_label"].mean().reindex(["high", "medium", "low"])
print("\nObserved decline-label rate by confidence tier (full scored population -- see Section 2 for the held-out validated number):")
display(decline_rate_by_conf.round(3))

archetype_map = pd.DataFrame([
    {"reason_code": "thin_visible_page", "meaning": "visible page (>=250 impr/90d), under 1200 words", "maps_to_action": "expand_and_refresh"},
    {"reason_code": "low_ctr_visible_page + (model_decline_risk or declining_with_demand)", "meaning": "page 1-20 position, CTR<0.5%, plus a decline signal", "maps_to_action": "refresh_and_review_ctr"},
    {"reason_code": "engagement_review_candidate + (model_decline_risk or declining_with_demand)", "meaning": ">=30 sessions/90d with weak engagement/scroll, plus a decline signal", "maps_to_action": "refresh_and_review_engagement"},
    {"reason_code": "model_decline_risk / declining_with_demand / stale_visible_page / visible_model_opportunity", "meaning": "one decline-or-opportunity signal, no sharper match above", "maps_to_action": "refresh"},
    {"reason_code": "(none cleared)", "meaning": "no rule or model threshold tripped", "maps_to_action": "monitor"},
])
print("\nArchetype -> action mapping:")
display(archetype_map)

decay_view = pd.crosstab(queue["freshness_tier"], queue["trend_direction"], normalize="index").round(3)
print("\nShare of trend_direction within each freshness_tier (row-normalized, observed in this snapshot):")
display(decay_view)

print("\nTop 5 of the ranked queue (reviewer's first look):")
display(queue[["final_rank", "final_refresh_score", "confidence", "suggested_action", "final_reason_codes", "impressions_90d", "trend_direction"]].head(5))

Suggested-action mix (n = 30,000 scored rows):


,count
suggested_action,
monitor,13083
refresh,8188
refresh_and_review_ctr,6654
refresh_and_review_engagement,1993
expand_and_refresh,82



Confidence mix:


,count
confidence,
high,3602
medium,11398
low,15000



Action x confidence (counts):


...

,reason_code,meaning,maps_to_action
0,thin_visible_page,"visible page (>=250 impr/90d), under 1200 words",expand_and_refresh
1,low_ctr_visible_page + (model_decline_risk or ...,"page 1-20 position, CTR<0.5%, plus a decline s...",refresh_and_review_ctr
2,engagement_review_candidate + (model_decline_r...,">=30 sessions/90d with weak engagement/scroll,...",refresh_and_review_engagement
3,model_decline_risk / declining_with_demand / s...,"one decline-or-opportunity signal, no sharper ...",refresh
4,(none cleared),no rule or model threshold tripped,monitor



Share of trend_direction within each freshness_tier (row-normalized, observed in this snapshot):


trend_direction,down,flat,new,stable,up
freshness_tier,,,,,
0-30,0.511,0.044,0.104,0.186,0.155
181+,0.471,0.092,0.144,0.138,0.155
31-90,0.589,0.006,0.040,0.149,0.217
91-180,0.611,0.026,0.008,0.229,0.126



Top 5 of the ranked queue (reviewer's first look):


,final_rank,final_refresh_score,confidence,suggested_action,final_reason_codes,impressions_90d,trend_direction
0,1,81.734212,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,12834,down
1,2,81.603243,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2498,down
2,3,81.544618,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,8064,down
3,4,81.169731,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,13790,down
4,5,80.957565,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,3393,down


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** This queue is a *prioritization aid* for a human content reviewer deciding which existing pages to look at first in a review cycle. It ranks; it does not decide. The honest evidence behind that ranking, carried over from ML-09's validation audit (`work/notebooks/w06_validation_audit.ipynb`, re-derived below so this notebook doesn't hand-type someone else's numbers): in a **client-grouped, held-out split** (no client appears in both train and test), the random-forest model reached a measured `precision@50` of 0.740 against a held-out base rate of 0.391 — a directional improvement over the transparent baseline rule's `precision@50` of 0.240. That is decision support for review order, not a verdict on any single page.

**I'm also naming the number I'm *not* using.** A naive random row split on the same data showed an inflated `precision@50` of 0.900, because 31 of 32 clients leaked across train and test — the model was partly recognizing clients it had already seen. That number is real but not trustworthy, and it doesn't appear anywhere else in this playbook.

**Limits, stated plainly:**

- **Scope.** One anonymized snapshot, 30,000 rows, 32 clients, a trailing-90-day window. This is not validated on any client, content type, or time period outside this slice.
- **Small held-out test set.** The client-grouped validation held out only 6 clients. That's enough to catch leakage, not enough to call the 0.740 number precise — treat it as a directional signal, not a guarantee.
- **The label is a same-window read, not an outcome.** `is_declining_label` comes from `trend_direction`, itself a 30-day-vs-prior-30-day comparison inside this snapshot. This is a decline-*risk* ranker. Nothing here has been validated against what happens *after* a page is actually refreshed — there's no controlled or before/after design in this dataset.
- **Uneven feature coverage by content type.** `feedly article` rows are missing `search_volume`, `competition`, and `cpc` entirely (100% missing for that type), and `keyword article` rows are missing `word_count` about 28% of the time. Recommendations on these rows are working from a smaller feature set than the rest of the queue.
- **`avg_position == 0` means "no rank data," not "rank zero."** These rows exist in the queue and need a reviewer's eye, not an automated read of position.
- **Coverage gap on quiet pages.** ML-09's error audit found the model's false negatives were low-impression pages (1-3 impressions/90d) — there's too little signal for the model to work with, so real declines on very quiet pages can slip through.
- **Client-specific miscalibration risk.** The same audit found both false positives landed on one held-out client the model hadn't trained on — a reminder that scores may run high or low simply because a client's normal profile differs from the training portfolio (see Section 3 and 4 for how this is handled operationally).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Re-derive the client-grouped honest benchmark inline (same method as work/notebooks/w06_validation_audit.ipynb)
# so this notebook's claims are backed by a number it computed itself, not a copied one.

sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES  # noqa: E402
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top = np.asarray(y_true)[order][:k]
    return float(top.mean())

feat = raw.copy()
raw_numeric_cols = [c for c in MODEL_NUMERIC_FEATURES if c in feat.columns]
for c in raw_numeric_cols:
    feat[c] = pd.to_numeric(feat[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
feat = feat[(feat["impressions_90d"] > 0) & (feat["content_age_days"] >= 90)].drop_duplicates(subset=["content_id"]).reset_index(drop=True)
feat["is_declining_label"] = feat["trend_direction"].str.lower().eq("down").astype(int)
feat["log_impressions_90d"] = np.log1p(feat["impressions_90d"])
feat["log_clicks_90d"] = np.log1p(feat["clicks_90d"])
feat["log_sessions_90d"] = np.log1p(feat["sessions_90d"])
feat["log_ai_sessions_90d"] = np.log1p(feat["ai_sessions_90d"])
for c in MODEL_CATEGORICAL_FEATURES:
    feat[c] = feat[c].fillna("unknown").astype(str)

y_full = feat["is_declining_label"]
numeric = feat[[c for c in MODEL_NUMERIC_FEATURES if c != "trend_pct"]].copy()
cat_enc = pd.get_dummies(feat[MODEL_CATEGORICAL_FEATURES], prefix=MODEL_CATEGORICAL_FEATURES, dtype=float)
X_full = pd.concat([numeric.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)

client_series = feat["client_id"].astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled) * 0.2)))
test_clients_set = set(shuffled[:test_client_count])
test_mask = client_series.isin(test_clients_set).to_numpy()
all_idx = np.arange(len(feat))
train_idx, test_idx = all_idx[~test_mask], all_idx[test_mask]
overlap = len(set(feat.iloc[train_idx]["client_id"]) & set(feat.iloc[test_idx]["client_id"]))
assert overlap == 0, "client leakage across the grouped split"

clf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                              n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
clf.fit(X_full.iloc[train_idx], y_full.iloc[train_idx])
prob = clf.predict_proba(X_full.iloc[test_idx])[:, 1]

client_grouped_benchmark = {
    "split": "client-grouped holdout",
    "client_overlap": overlap,
    "test_clients": int(feat.iloc[test_idx]["client_id"].nunique()),
    "roc_auc": round(roc_auc_score(y_full.iloc[test_idx], prob), 3),
    "avg_precision": round(average_precision_score(y_full.iloc[test_idx], prob), 3),
    "precision_at_50": round(precision_at_k(y_full.iloc[test_idx], prob, 50), 3),
    "held_out_base_rate": round(float(y_full.iloc[test_idx].mean()), 3),
    "baseline_rule_precision_at_50": model_results["baseline"]["baseline_precision_at_50"],
}
print("Client-grouped validation benchmark (re-derived here, matches work/notebooks/w06_validation_audit.ipynb):")
display(pd.Series(client_grouped_benchmark))

# Real numbers behind the "uneven feature coverage" and "no rank data" limits
feedly_share = (raw["content_type"] == "feedly article").mean()
feedly_missing = raw.loc[raw["content_type"] == "feedly article", ["search_volume", "competition", "cpc"]].isna().mean()
keyword_wc_missing = raw.loc[raw["content_type"] == "keyword article", "word_count"].isna().mean()
zero_position_rows = int((raw["avg_position"] == 0).sum())

print(f"\nfeedly article: {feedly_share:.1%} of rows, missing search_volume/competition/cpc: {feedly_missing.round(3).to_dict()}")
print(f"keyword article word_count missing: {keyword_wc_missing:.1%}")
print(f"avg_position == 0 (no rank data) rows: {zero_position_rows:,}")

Client-grouped validation benchmark (re-derived here, matches work/notebooks/w06_validation_audit.ipynb):


,0
split,client-grouped holdout
client_overlap,0
test_clients,6
roc_auc,0.75
avg_precision,0.618
precision_at_50,0.74
held_out_base_rate,0.391
baseline_rule_precision_at_50,0.24



feedly article: 7.0% of rows, missing search_volume/competition/cpc: {'search_volume': 1.0, 'competition': 1.0, 'cpc': 1.0}
keyword article word_count missing: 28.3%
avg_position == 0 (no rank data) rows: 1,205


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any single row, a reviewer checks:**

1. **Read the page.** Reason codes are a starting point, not a verdict — confirm the flagged issue (thin content, weak CTR, stale update) still matches what's actually on the page today.
2. **Check `content_type`.** `feedly article` rows are working from a smaller feature set (no keyword/competition/cpc data) — treat these as needing a closer look, not a lighter one.
3. **Watch for one client dominating a batch.** ML-09's audit found the model's false positives clustered on a single held-out client — if a review batch is mostly one client, don't rubber-stamp it; that client's normal profile may not match the training portfolio.
4. **Don't read `monitor` as "healthy."** It means no rule or model threshold tripped, not that the page was checked and cleared (see the fallback-rate check below).
5. **Confirm the row was actually in scope.** Only pages with `impressions_90d > 0` and `content_age_days >= 90` were scored at all — very new or completely invisible pages aren't represented here.

**No-go list — never automate:**

- **No auto-publishing or auto-editing.** A human writes and approves every content change; this queue only orders the review list.
- **No causal claims to clients.** "Refreshing this will recover X% of traffic" is not supported by this design — there's no before/after or controlled comparison here, only a same-snapshot decline-risk ranking.
- **No cross-client leaderboard.** Scores aren't client-normalized; comparing `final_refresh_score` across clients would mostly measure traffic scale, not need.
- **No trusting the `expand_and_refresh` bucket as validated on its own** — it's a small slice of the queue (see count below); review these individually rather than batch-approving.
- **No reusing `trend_direction` / `trend_pct` downstream as if it were new model insight** — it's the exact field the label is built from; treating it as a separate finding is circular.
- **No skipping review because `confidence == "high"`.** Confidence reflects the score plus a traffic/session volume floor — not a claim that the model is verified correct on that specific row.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

expand_n = int((queue["suggested_action"] == "expand_and_refresh").sum())
expand_share = expand_n / len(queue)
print(f"expand_and_refresh: {expand_n} rows ({expand_share:.1%} of the queue) -- too small to treat as a separately validated action; review each one.")

fallback_share = queue["final_reason_codes"].str.contains("general_refresh_review").mean()
print(f"general_refresh_review fallback share: {fallback_share:.1%} of the queue")
print("(this is the 'no rule or model threshold tripped' catch-all -- it should be near the 'monitor' count, not a hidden third category)")

monitor_no_reason = queue[(queue["suggested_action"] == "monitor")]["final_reason_codes"].str.contains("general_refresh_review").mean()
print(f"Of rows suggested_action == 'monitor', share tagged general_refresh_review: {monitor_no_reason:.1%}")

feedly_conf = queue.loc[queue["content_type"] == "feedly article", "confidence"].value_counts(normalize=True).round(3)
print("\nConfidence mix within feedly article rows (the reduced-feature-coverage content type):")
display(feedly_conf)

# Which single client contributes the most rows to any one action -- a batch-concentration check
top_client_share_by_action = (
    queue.groupby("suggested_action")["client_id"]
    .apply(lambda s: s.value_counts(normalize=True).iloc[0] if len(s) else 0.0)
    .round(3)
)
print("\nLargest single-client share within each suggested_action (flag a batch if this is high):")
display(top_client_share_by_action)

expand_and_refresh: 82 rows (0.3% of the queue) -- too small to treat as a separately validated action; review each one.
general_refresh_review fallback share: 30.4% of the queue
(this is the 'no rule or model threshold tripped' catch-all -- it should be near the 'monitor' count, not a hidden third category)
Of rows suggested_action == 'monitor', share tagged general_refresh_review: 58.9%

Confidence mix within feedly article rows (the reduced-feature-coverage content type):


,proportion
confidence,
low,0.855
medium,0.132
high,0.012



Largest single-client share within each suggested_action (flag a batch if this is high):


,client_id
suggested_action,
expand_and_refresh,0.341
monitor,0.235
refresh,0.180
refresh_and_review_ctr,0.347
refresh_and_review_engagement,0.598


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

None of this is production monitoring infrastructure — it's a lightweight, practical set of checks a small team could actually run each cycle, following the pattern in Evidently's data-drift guidance (compare a fresh pull against the training snapshot's own distribution, not against a fixed "good" number).

- **Population drift.** Each new pull, compare the median `impressions_90d`, median `avg_position` (rank-known rows only), and median `days_since_last_update` against this run's baseline (computed below). A shift of roughly 20%+ in any of these medians means the portfolio being scored looks meaningfully different from the one the model was trained on — investigate before trusting new scores.
- **Action/confidence-mix drift.** Track the `suggested_action` and `confidence` mix each run. A material swing (e.g. the high-confidence share moving far from this run's baseline) signals the model is scoring a different kind of content than it saw at training time.
- **Precision decay (the real check).** Periodically — quarterly is reasonable — pull a fresh sample of `confidence == "high"` rows, have an editor mark the actual outcome after review, and compare the hit-rate against the 0.740 `precision@50` benchmark from Section 2. A meaningful drop means: pause treating this as decision support and re-validate before relying on it again.
- **New-client rule.** Any client not among the 32 in this training run defaults to `confidence = "low"` regardless of its raw score, until enough history exists to check the model isn't just extrapolating from unrelated clients — directly motivated by the single-client false-positive pattern in Section 3.
- **Retrain cadence.** Quarterly, or immediately if any trigger above fires. Retraining more often than the underlying 90-day feature window changes doesn't add real signal, and it makes the precision-decay check (which needs a stable benchmark to compare against) harder to read.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

monitoring_baseline = {
    "median_impressions_90d": float(queue["impressions_90d"].median()),
    "median_avg_position_rank_known": float(queue.loc[queue["avg_position"] > 0, "avg_position"].median()),
    "median_days_since_last_update": float(queue["days_since_last_update"].median()),
    "action_mix_pct": (queue["suggested_action"].value_counts(normalize=True) * 100).round(2).to_dict(),
    "confidence_mix_pct": (queue["confidence"].value_counts(normalize=True) * 100).round(2).to_dict(),
    "drift_threshold_pct": 20,
    "precision_at_50_benchmark": client_grouped_benchmark["precision_at_50"],
    "held_out_base_rate_benchmark": client_grouped_benchmark["held_out_base_rate"],
    "retrain_cadence": "quarterly, or immediately on any trigger above",
    "trained_client_count": int(raw["client_id"].nunique()),
}
print("Monitoring baseline (this run's reference values -- future pulls compare against these):")
print(json.dumps(monitoring_baseline, indent=2))

Monitoring baseline (this run's reference values -- future pulls compare against these):
{
  "median_impressions_90d": 731.0,
  "median_avg_position_rank_known": 11.4,
  "median_days_since_last_update": 20.0,
  "action_mix_pct": {
    "monitor": 43.61,
    "refresh": 27.29,
    "refresh_and_review_ctr": 22.18,
    "refresh_and_review_engagement": 6.64,
    "expand_and_refresh": 0.27
  },
  "confidence_mix_pct": {
    "low": 50.0,
    "medium": 37.99,
    "high": 12.01
  },
  "drift_threshold_pct": 20,
  "precision_at_50_benchmark": 0.74,
  "held_out_base_rate_benchmark": 0.391,
  "retrain_cadence": "quarterly, or immediately on any trigger above",
  "trained_client_count": 32
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exporting the full ranked queue plus a compact, machine-readable playbook summary (the receipts: action mix, archetype mapping, the client-grouped validation benchmark, known limits, monitoring baseline, and the no-go list) to `work/outputs/`. The queue CSV is intentionally not committed to git (this notebook regenerates it from the bundled starter CSV every run); the JSON summary is committed, since it's the trail my paper's numbers trace back to. I'm also copying the three most reusable charts into `work/figures/` so the paper doesn't have to regenerate SVGs from scratch.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

WORK_OUTPUTS = ROOT / "work" / "outputs"
WORK_FIGURES = ROOT / "work" / "figures"
WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)
WORK_FIGURES.mkdir(parents=True, exist_ok=True)

export_cols = [
    "final_rank", "content_id", "client_id", "final_refresh_score", "confidence",
    "suggested_action", "final_reason_codes", "best_model_probability", "is_declining_label",
    "impressions_90d", "sessions_90d", "avg_position", "ctr", "content_age_days",
    "days_since_last_update", "word_count", "trend_direction", "content_type", "freshness_tier",
]
queue_path = WORK_OUTPUTS / "w07_ranked_queue.csv"
queue[export_cols].to_csv(queue_path, index=False)
print(f"Wrote {queue_path} ({len(queue):,} rows) -- gitignored by design, regenerated by this notebook.")

playbook_summary = {
    "source_dataset": "data/raw/content_refresh_anonymized.csv (30,000 rows, 32 clients, bundled starter slice)",
    "rows_scored": int(len(queue)),
    "best_model": model_results["best_model"]["name"],
    "action_mix": action_counts.to_dict(),
    "confidence_mix": {k: int(v) for k, v in confidence_counts.to_dict().items()},
    "archetype_to_action": archetype_map.to_dict(orient="records"),
    "validation_benchmark": client_grouped_benchmark,
    "naive_split_number_not_used": {
        "precision_at_50": 0.900,
        "why_excluded": "31 of 32 clients leaked across train/test on a naive random row split -- inflated, not trustworthy (see work/notebooks/w06_validation_audit.ipynb).",
    },
    "known_limits": {
        "feedly_article_share_pct": round(float(feedly_share) * 100, 1),
        "feedly_article_missing_keyword_fields": True,
        "keyword_article_word_count_missing_pct": round(float(keyword_wc_missing) * 100, 1),
        "avg_position_zero_rows": zero_position_rows,
        "expand_and_refresh_bucket_size": expand_n,
        "test_set_client_count": client_grouped_benchmark["test_clients"],
    },
    "monitoring_baseline": monitoring_baseline,
    "no_go_list": [
        "No auto-publishing or auto-editing from this queue -- a human writes/approves every content change.",
        "No causal or 'this will recover traffic' claims -- decision-support ranking only, not a controlled experiment.",
        "No cross-client score comparisons -- scores aren't client-normalized.",
        f"No treating expand_and_refresh as a separately validated action -- bucket too small (n={expand_n}).",
        "No reusing trend_direction/trend_pct downstream as if it were new model insight -- it's the label source.",
        "No skipping review because confidence == 'high' -- confidence reflects score + volume thresholds, not verified correctness.",
    ],
}
summary_path = WORK_OUTPUTS / "w07_playbook_summary.json"
summary_path.write_text(json.dumps(playbook_summary, indent=2))
print(f"Wrote {summary_path} (committed -- these are the receipts the paper cites)")

for chart in ["action_mix.svg", "confidence_mix.svg", "top_reason_codes.svg"]:
    src = ROOT / "outputs" / "charts" / chart
    dst = WORK_FIGURES / f"w07_{chart}"
    shutil.copyfile(src, dst)
    print(f"Copied {src.name} -> {dst}")

Wrote /content/flyrankAI-ML/work/outputs/w07_ranked_queue.csv (30,000 rows) -- gitignored by design, regenerated by this notebook.
Wrote /content/flyrankAI-ML/work/outputs/w07_playbook_summary.json (committed -- these are the receipts the paper cites)
Copied action_mix.svg -> /content/flyrankAI-ML/work/figures/w07_action_mix.svg
Copied confidence_mix.svg -> /content/flyrankAI-ML/work/figures/w07_confidence_mix.svg
Copied top_reason_codes.svg -> /content/flyrankAI-ML/work/figures/w07_top_reason_codes.svg


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.